# 2023 — Sparse Autoencoders: Reading Features out of a Network

**Papers:** Nelson Elhage et al., *"Toy Models of Superposition"* (Anthropic, 2022); Trenton Bricken et al., *"Towards Monosemanticity: Decomposing Language Models With Dictionary Learning"* (Anthropic, 2023); Hoagy Cunningham et al., *"Sparse Autoencoders Find Highly Interpretable Features in Language Models"* (2023); Adly Templeton et al., *"Scaling Monosemanticity"* (Anthropic, 2024). Dictionary learning itself goes back to Olshausen & Field (1996).

Every other notebook asks how to make networks better. This one asks **what they have learned**. Individual neurons are often **polysemantic**: one neuron fires for unrelated things. The **superposition hypothesis** explains why. A layer with $d$ neurons can represent many more than $d$ features as *directions* in activation space, as long as the features are **sparse** (rarely active at the same time). Nearly orthogonal directions interfere only a little, and the network accepts that small cost to store more.

If so, the features are recoverable by **dictionary learning**: find an overcomplete set of directions such that each activation vector is a sparse combination of them. A **sparse autoencoder (SAE)** does this:

$$f = \operatorname{ReLU}(W_{enc}(x - b_{dec}) + b_{enc}), \qquad \hat x = W_{dec} f + b_{dec}, \qquad \mathcal L = \lVert x-\hat x\rVert^2 + \lambda\lVert f\rVert_1$$

with many more latents than input dimensions. In 2024, SAEs with millions of latents on Claude 3 Sonnet found features for concepts such as the Golden Gate Bridge, code vulnerabilities and sycophancy, and turning them up or down steered the model's behaviour.

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.15.0 | device: mps


In [2]:
class SAE(nn.Module):
    """ReLU SAE with an L1 penalty, or (k given) a TopK SAE that keeps only the k largest latents (Gao et al., 2024)."""
    def __init__(self, d, n_latents, k=None):
        super().__init__()
        self.W_enc = nn.Parameter(torch.randn(n_latents, d) * 0.1); self.b_enc = nn.Parameter(torch.zeros(n_latents))
        self.W_dec = nn.Parameter(self.W_enc.data.T.clone()); self.b_dec = nn.Parameter(torch.zeros(d)); self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc.T + self.b_enc
        if self.k is None: return F.relu(pre)
        v, i = pre.topk(self.k, dim=-1)
        return torch.zeros_like(pre).scatter(-1, i, F.relu(v))
    def forward(self, x):
        f = self.encode(x); return f @ self.W_dec.T + self.b_dec, f
    def normalize(self):
        with torch.no_grad(): self.W_dec /= self.W_dec.norm(dim=0, keepdim=True)  # unit-norm dictionary directions

def train_sae(acts, n_latents, l1=0.0, k=None, steps=8000, lr=1e-3, B=512, seed=0):
    seed_everything(seed); sae = SAE(acts.shape[1], n_latents, k).to(acts.device); opt = torch.optim.Adam(sae.parameters(), lr=lr)
    for s in range(steps):
        x = acts[torch.randint(0, len(acts), (B,), device=acts.device)]
        xh, f = sae(x); loss = (xh - x).pow(2).sum(-1).mean() + l1 * f.abs().sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sae.normalize()
    with torch.no_grad():
        xh, f = sae(acts); fve = 1 - (xh - acts).pow(2).sum() / (acts - acts.mean(0)).pow(2).sum()
    return sae, fve.item(), (f > 1e-3).float().sum(-1).mean().item()

## Part 1: superposition in a controlled setting
Make activations exactly as the hypothesis describes: 64 ground-truth features, stored as random directions in a **16-dimensional** space. Each example activates each feature independently with probability 3%, with a random positive magnitude. Can we get the 64 directions back from the 16-d activations alone?

In [3]:
seed_everything(0)
d, n_feat = 16, 64
true_dirs = F.normalize(torch.randn(n_feat, d), dim=-1).to(device)
def sample_acts(n, p=0.03):
    on = (torch.rand(n, n_feat, device=device) < p).float()
    return (on * torch.rand(n, n_feat, device=device)) @ true_dirs
acts = sample_acts(100_000)
print(f"{n_feat} features in {d} dimensions; max |cos| between two true directions {(true_dirs @ true_dirs.T - torch.eye(n_feat, device=device)).abs().max():.2f}")

def recovered(dirs, thr=0.9):
    """For each true feature: best |cosine| with any learned direction; count how many exceed thr."""
    cos = (F.normalize(dirs, dim=-1) @ true_dirs.T).abs().max(0).values
    return (cos > thr).sum().item(), cos.mean().item()
U, S_, Vt = torch.linalg.svd(acts - acts.mean(0), full_matrices=False)
print(f"PCA ({d} components):        recovers {recovered(Vt)[0]:2d}/{n_feat} features (mean best cosine {recovered(Vt)[1]:.2f})")
neurons = torch.eye(d, device=device)
print(f"the {d} neurons themselves:  recovers {recovered(neurons)[0]:2d}/{n_feat} features (mean best cosine {recovered(neurons)[1]:.2f})")
for l1, k in [(0.0, None), (0.03, None), (0.3, None), (0.0, 2), (0.0, 4), (0.0, 8)]:
    sae, fve, l0 = train_sae(acts, 128, l1=l1, k=k)
    n, m = recovered(sae.W_dec.T)
    name = f"L1 SAE, λ = {l1:<4}" if k is None else f"TopK SAE, k = {k}  "
    print(f"{name}: recovers {n:2d}/{n_feat} features (mean best cosine {m:.2f}) | variance explained {fve:.3f} | active latents per input {l0:.1f}")

64 features in 16 dimensions; max |cos| between two true directions 0.76
PCA (16 components):        recovers  0/64 features (mean best cosine 0.55)
the 16 neurons themselves:  recovers  0/64 features (mean best cosine 0.54)


/var/folders/wc/dmtxvwwd4yd6x5qrr6981_v40000gn/T/ipykernel_6265/4008028315.py:14: UserWarning: linalg.svd: matrix too large to stage in MPS threadgroup memory (6400000 > 32768 bytes); falling back to CPU. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/mps/operations/LinearAlgebra.mm:2288.)
  U, S_, Vt = torch.linalg.svd(acts - acts.mean(0), full_matrices=False)


L1 SAE, λ = 0.0 : recovers 40/64 features (mean best cosine 0.90) | variance explained 1.000 | active latents per input 62.5


L1 SAE, λ = 0.03: recovers  2/64 features (mean best cosine 0.77) | variance explained 0.992 | active latents per input 33.6


L1 SAE, λ = 0.3 : recovers 48/64 features (mean best cosine 0.93) | variance explained 0.808 | active latents per input 7.1


TopK SAE, k = 2  : recovers 64/64 features (mean best cosine 1.00) | variance explained 0.855 | active latents per input 2.0


TopK SAE, k = 4  : recovers 64/64 features (mean best cosine 0.98) | variance explained 0.904 | active latents per input 4.0


TopK SAE, k = 8  : recovers 25/64 features (mean best cosine 0.89) | variance explained 0.942 | active latents per input 8.0


PCA can only return 16 orthogonal directions, and the raw neurons are mixtures, so neither finds the features. The SAEs show that **the amount of sparsity is what matters**. The data has about 2 active features per input. A TopK SAE that keeps exactly 2 (or 4) latents recovers every planted direction, at the cost of reconstructing a bit less of the variance; keeping 8 is already too loose. With an L1 penalty, a small $\lambda$ leaves dozens of latents active and recovers *fewer* features than no penalty at all; only a large $\lambda$ pushes activity down to a few latents and recovers most of them. Reconstruction quality alone is a poor guide: the best reconstructions come from the least interpretable dictionaries.
## Part 2: features inside a small language model
Train a small character-level GPT on the procedural corpus, collect its MLP activations (512-d) in the last block, and train a TopK SAE with 2,048 latents (16 active per position) on them. Then, for a few latents, list the contexts where they fire most strongly.

In [4]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")

290,321 characters, vocab 31


In [5]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj, self.h = nn.Linear(d, 3 * d), nn.Linear(d, d), heads
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    """A small GPT-2-style character model (pre-norm blocks, learned positions)."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return self.head(self.ln(x))

def train_gpt(model, train_data, steps=800, B=32, lr=2e-3):
    model = model.to(train_data.device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - model.ctx - 1, (B,), device=train_data.device)
        x = train_data[ix[:, None] + torch.arange(model.ctx, device=train_data.device)]; y = train_data[ix[:, None] + torch.arange(1, model.ctx + 1, device=train_data.device)]
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def eval_loss(model, data, n=20, B=32):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    for _ in range(n):
        ix = torch.randint(0, len(data) - model.ctx - 1, (B,), generator=g).to(data.device)
        x = data[ix[:, None] + torch.arange(model.ctx, device=data.device)]; y = data[ix[:, None] + torch.arange(1, model.ctx + 1, device=data.device)]
        losses.append(F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1)).item())
    return float(np.mean(losses))

In [6]:
seed_everything(0); t0 = time.time()
gpt = train_gpt(GPT(V, ctx=64), train_data, steps=1000); print(f"GPT val loss {eval_loss(gpt, val_data):.3f} ({time.time() - t0:.0f}s)")
store = {}
hook = gpt.blocks[-1].mlp[1].register_forward_hook(lambda m, i, o: store.__setitem__("h", o.detach()))
with torch.no_grad():
    starts = torch.arange(0, 60_000, 64, device=device)
    windows = torch.stack([data[s:s + 64] for s in starts.tolist()])
    gpt(windows); H = store["h"].reshape(-1, store["h"].shape[-1])          # one activation vector per character position
hook.remove()
print(f"{len(H):,} activation vectors of size {H.shape[1]}")
H = H / H.norm(dim=-1).mean()                                                # normalise scale so λ is meaningful
sae, fve, l0 = train_sae(H, 2048, k=16, steps=6000)
with torch.no_grad(): Fz = sae.encode(H)
alive = (Fz > 0).any(0)
print(f"SAE: variance explained {fve:.3f}, {l0:.1f} active latents per position on average, {alive.sum().item()} of 2048 latents ever active")

GPT val loss 0.238 (35s)
60,032 activation vectors of size 512


SAE: variance explained 0.979, 16.0 active latents per position on average, 1587 of 2048 latents ever active


In [7]:
flat = windows.reshape(-1)
def top_contexts(j, k=6, width=24):
    vals, pos = Fz[:, j].topk(k)
    out = []
    for v, p in zip(vals.tolist(), pos.tolist()):
        a = max(0, p - width); s = decode(flat[a:p].tolist()); c = decode([flat[p].item()])
        out.append(f"  {v:5.2f}  …{s}[{c}]")
    return out
freq = (Fz > 0).float().mean(0)
cands = torch.where((freq > 0.002) & (freq < 0.05))[0]                         # latents that fire neither always nor never
seed_everything(1); show = cands[torch.randperm(len(cands))[:6]].tolist()
for j in show:
    print(f"latent {j} (active on {freq[j]:.1%} of positions); strongest activations, the [character] is where it fires:")
    print("\n".join(top_contexts(j)))

latent 371 (active on 2.2% of positions); strongest activations, the [character] is where it fires:
   0.16  …, then the teacher forge[t]
   0.15  …e loss. The router forge[t]
   0.15  …rallel. The router forge[t]
   0.15  … loss. Each layer predic[t]
   0.15  …gets its weights step by[ ]
   0.15  …token. The teacher forge[t]
latent 873 (active on 1.8% of positions); strongest activations, the [character] is where it fires:
   0.22  …e carefully. The attenti[o]
   0.22  …outer remembers a long c[o]
   0.22  … next token. The attenti[o]
   0.22  … again, then the attenti[o]
   0.22  …owly because the attenti[o]
   0.21  …in parallel. The attenti[o]
latent 1299 (active on 1.6% of positions); strongest activations, the [character] is where it fires:
   0.15  …forgets the key and valu[e]
   0.15  …forgets the key and valu[e]
   0.15  …the key and value withou[t]
   0.14  …r copies the gradient on[c]
   0.14  …ets the key and value on[c]
   0.14  …redicts the key and valu[e]
latent 1109 (ac

## How monosemantic are the latents? Compare with raw neurons
A rough automatic test: for each unit, take its 50 strongest activations and ask how often the **next character to be predicted** is the same. A unit that encodes one thing ("about to write 'carefully'") should score high; a polysemantic neuron mixes several.

In [8]:
nxt = torch.cat([flat[1:], flat[:1]])
def purity(acts_matrix, units):
    scores = []
    for j in units:
        idx = acts_matrix[:, j].topk(50).indices
        scores.append(torch.bincount(nxt[idx], minlength=V).max().item() / 50)
    return np.mean(scores)
neuron_ids = torch.randperm(H.shape[1])[:200].tolist(); latent_ids = cands[torch.randperm(len(cands))[:200]].tolist()
print(f"next-character purity of top activations: raw MLP neurons {purity(H, neuron_ids):.2f} | SAE latents {purity(Fz, latent_ids):.2f}")

next-character purity of top activations: raw MLP neurons 0.77 | SAE latents 0.85


## Takeaways
- **Superposition:** networks store more features than they have neurons, as sparse, nearly orthogonal directions. That is why single neurons are hard to interpret.
- **Sparse dictionary learning recovers them, if the sparsity is right.** In the controlled test a TopK SAE with $ near the true number of active features recovers all 64 planted directions, which PCA and the raw neurons cannot; a weak L1 penalty does worse than none. On the small LM, TopK latents are purer than raw neurons by a simple next-character test (0.85 vs. 0.77), and individual latents read cleanly (one fires on "the next [t]", another on the space in "then the[ ]"). This corpus is highly templated, so treat that as a sanity check, not evidence about large models.
- **Caveats** (all active research): the number of latents and λ change which features you get ("feature splitting"); some reconstruction error remains unexplained ("dark matter"); and an interpretable latent is a hypothesis to test, for example by **steering** (adding the decoder direction to the activations) and checking that behaviour changes as predicted. Variants in use include TopK SAEs (OpenAI, 2024), JumpReLU SAEs (Gemma Scope, 2024) and transcoders / cross-layer transcoders for circuit tracing (Anthropic, 2025).